# DATA 255 - Lab 3
### LiteSRNet - Lightweight Super Resolution Network
### Goal: PSNR > 24.3 dB, Low NPU Latency
### Input/Output: [256, 256, 3]

In [ ]:

import subprocess, sys, os

print('Installing required libraries...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'onnx', 'onnxscript', '--quiet'])

import gc, time, random, math, shutil, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from PIL import Image
from google.colab import drive

# ── Mount Drive ──────────────────────────────────────────────────────────────
drive.mount('/content/drive')

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT     = '/content/drive/MyDrive/Lab3'
DRIVE_SAVE    = '/content/drive/MyDrive/Lab3/outputs_litesr'
CHECKPOINT_DIR = '/content/checkpoints'
ONNX_PATH     = '/content/sr_model.onnx'
os.makedirs(DRIVE_SAVE, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Extract Datasets ─────────────────────────────────────────────────────────
zip_pairs = [
    ('L3_HR_train.zip',  '/content/HR_train'),
    ('L3_LR_train.zip',  '/content/LR_train'),
    ('L3_HR_valid.zip',  '/content/HR_val'),
    ('L3_LR_valid.zip',  '/content/LR_val'),
    ('L3_HR_train1.zip', '/content/HR_train1'),
    ('L3_LR_train1.zip', '/content/LR_train1'),
    ('L3_HR_train2.zip', '/content/HR_train2'),
    ('L3_LR_train2.zip', '/content/LR_train2'),
    ('L3_HR_train3.zip', '/content/HR_train3'),
    ('L3_LR_train3.zip', '/content/LR_train3'),
    ('L3_HR_train4.zip', '/content/HR_train4'),
    ('L3_LR_train4.zip', '/content/LR_train4'),
]

print('Extracting datasets...')
for _zip, _dest in zip_pairs:
    zip_path = os.path.join(DATA_ROOT, _zip)
    if not os.path.exists(zip_path):
        continue
    os.makedirs(_dest, exist_ok=True)
    r = subprocess.run(['unzip', '-q', '-o', zip_path, '-d', _dest], capture_output=True)
    if r.returncode != 0:
        time.sleep(3)
        subprocess.run(['unzip', '-q', '-o', zip_path, '-d', _dest], capture_output=True)
print('Done. Extracted 12 zip files.\n')

# ── Device ───────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Installing required libraries...
Mounted at /content/drive
Extracting datasets...
Done. Extracted 12 zip files.

Device : cuda
GPU    : NVIDIA A100-SXM4-40GB


In [ ]:
# ── Image utilities ───────────────────────────────────────────────────────────
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif'}

def find_images(root):
    paths = []
    for dirpath, _, files in os.walk(root):
        for f in sorted(files):
            if os.path.splitext(f)[1].lower() in IMAGE_EXTS:
                paths.append(os.path.join(dirpath, f))
    return sorted(paths)


# ── Dataset ───────────────────────────────────────────────────────────────────
class SRDataset(Dataset):
    """
    Paired LR/HR dataset for image restoration.
    Input/Output: 256x256x3, normalized to [0,1]
    """
    def __init__(self, lr_paths, hr_paths, augment=False):
        assert len(lr_paths) == len(hr_paths)
        self.lr_paths = lr_paths
        self.hr_paths = hr_paths
        self.augment  = augment
        self.size     = 256

    def __len__(self):
        return len(self.lr_paths)

    def _load(self, path):
        img = Image.open(path).convert('RGB')
        if img.size != (self.size, self.size):
            img = img.resize((self.size, self.size), Image.BILINEAR)
        return img

    def __getitem__(self, idx):
        lr = self._load(self.lr_paths[idx])
        hr = self._load(self.hr_paths[idx])

        if self.augment:
            if random.random() > 0.5:
                lr = TF.hflip(lr); hr = TF.hflip(hr)
            if random.random() > 0.5:
                lr = TF.vflip(lr); hr = TF.vflip(hr)
            if random.random() > 0.3:
                k = random.choice([1, 2, 3])
                lr = TF.rotate(lr, k * 90)
                hr = TF.rotate(hr, k * 90)
            if random.random() > 0.5:
                f = random.uniform(0.85, 1.15)
                lr = TF.adjust_brightness(lr, f)
                hr = TF.adjust_brightness(hr, f)

        lr_t = torch.from_numpy(np.array(lr, dtype=np.float32) / 255.0).permute(2, 0, 1)
        hr_t = torch.from_numpy(np.array(hr, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return lr_t, hr_t


# ── Load all training data ────────────────────────────────────────────────────
print('Loading ALL training images...')

lr_train_roots = [
    '/content/LR_train', '/content/LR_train1', '/content/LR_train2',
    '/content/LR_train3', '/content/LR_train4'
]
hr_train_roots = [
    '/content/HR_train', '/content/HR_train1', '/content/HR_train2',
    '/content/HR_train3', '/content/HR_train4'
]

all_lr, all_hr = [], []
for lr_root, hr_root in zip(lr_train_roots, hr_train_roots):
    lrs = find_images(lr_root)
    hrs = find_images(hr_root)
    min_len = min(len(lrs), len(hrs))
    all_lr.extend(lrs[:min_len])
    all_hr.extend(hrs[:min_len])

lr_val_paths = find_images('/content/LR_val')
hr_val_paths = find_images('/content/HR_val')

train_dataset = SRDataset(all_lr, all_hr, augment=True)
val_dataset   = SRDataset(lr_val_paths, hr_val_paths, augment=False)

print(f'Train pairs : {len(train_dataset)}')
print(f'Val pairs   : {len(val_dataset)}')

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE    = 256
BATCH_SIZE  = 16
NUM_EPOCHS  = 350
GRAD_CLIP   = 1.0
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=1,
    shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True
)
n_batches = len(train_loader)
print(f'Train batches: {n_batches}')

Loading ALL training images...
Train pairs : 2217
Val pairs   : 110
Train batches: 138


In [ ]:
# ── Model Architecture ────────────────────────────────────────────────────────

class ConvBlock(nn.Module):
    """
    Basic conv block: Conv -> BN -> ReLU.
    NPU friendly: no complex activations.
    """
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=padding, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.act  = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class ResBlock(nn.Module):
    """
    Residual block with two conv layers.
    ReLU only activation for NPU compatibility.
    No BN in residual path for stable training.
    """
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=True)
        self.act   = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=True)
        # small init for stable residual learning
        nn.init.normal_(self.conv2.weight, std=0.01)
        nn.init.zeros_(self.conv2.bias)

    def forward(self, x):
        return x + self.conv2(self.act(self.conv1(x)))


class LiteSRNet(nn.Module):
    """
    Lightweight Super Resolution Network.
    Input : [B, 3, 256, 256] degraded LR image
    Output: [B, 3, 256, 256] restored HR image

    Architecture:
    1. Stride-2 conv: 256x256 -> 128x128 (process at half resolution)
    2. Feature extraction + BN normalization
    3. Deep residual body at 128x128
    4. ConvTranspose upsample: 128x128 -> 256x256
    5. Global residual: learn only the correction

    NPU optimizations:
    - ReLU only (no GELU, Sigmoid etc)
    - Standard Conv + ConvTranspose only
    - No skip concatenation (additive only)
    - Fixed spatial dimensions
    """
    def __init__(self, ch=56, num_blocks=9):
        super().__init__()

        # Encoder: 256x256 -> 128x128
        self.encoder = nn.Sequential(
            nn.Conv2d(3, ch, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(inplace=True)
        )

        # Feature normalization
        self.feat_norm = ConvBlock(ch, ch, kernel_size=3, stride=1, padding=1)

        # Deep residual body at 128x128
        self.body = nn.Sequential(
            *[ResBlock(ch) for _ in range(num_blocks)]
        )

        # Body projection
        self.body_proj = nn.Conv2d(ch, ch, 3, padding=1, bias=True)

        # Decoder: 128x128 -> 256x256
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(ch, ch, kernel_size=2, stride=2, bias=True),
            nn.ReLU(inplace=True)
        )

        # Output: reconstruct correction
        self.tail = nn.Conv2d(ch, 3, kernel_size=3, padding=1, bias=True)

        # Small init for tail -> starts as near-identity
        nn.init.normal_(self.tail.weight, std=0.001)
        nn.init.zeros_(self.tail.bias)

    def forward(self, x):
        # Encode
        e = self.encoder(x)              # 128x128
        f = self.feat_norm(e)            # 128x128

        # Process
        b = self.body_proj(self.body(f)) + f  # 128x128 with residual

        # Decode
        d = self.decoder(b)              # 256x256

        # Global residual learning
        return torch.clamp(x + self.tail(d), 0.0, 1.0)


# Build model
model = LiteSRNet(ch=56, num_blocks=9).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: LiteSRNet  |  Parameters: {n_params:,}')

# Verify input/output shape
with torch.no_grad():
    dummy = torch.randn(1, 3, 256, 256, device=device)
    out   = model(dummy)
    print(f'Input  : {list(dummy.shape)}')
    print(f'Output : {list(out.shape)}')

# GPU latency benchmark
print('\nMeasuring GPU latency...')
model.eval()
with torch.no_grad():
    for _ in range(30):
        _ = model(dummy)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(200):
        _ = model(dummy)
    torch.cuda.synchronize()
    gpu_lat = (time.perf_counter() - t0) / 200 * 1000

print(f'GPU Latency : {gpu_lat:.2f} ms/image')
print(f'Est. NPU    : ~{gpu_lat * 5:.1f} to {gpu_lat * 6:.1f} ms/image')
model.train()

Model: LiteSRNet  |  Parameters: 581,339
Input  : [1, 3, 256, 256]
Output : [1, 3, 256, 256]

Measuring GPU latency...
GPU Latency : 1.79 ms/image
Est. NPU    : ~8.9 to 10.7 ms/image


LiteSRNet(
  (encoder): Sequential(
    (0): Conv2d(3, 56, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU(inplace=True)
  )
  (feat_norm): ConvBlock(
    (conv): Conv2d(56, 56, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(56, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): ReLU(inplace=True)
  )
  (body): Sequential(
    (0): ResBlock(
      (conv1): Conv2d(56, 56, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (act): ReLU(inplace=True)
      (conv2): Conv2d(56, 56, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (1): ResBlock(
      (conv1): Conv2d(56, 56, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (act): ReLU(inplace=True)
      (conv2): Conv2d(56, 56, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (2): ResBlock(
      (conv1): Conv2d(56, 56, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (act): ReLU(inplace=True)
      (conv2): Conv2d(56, 

In [ ]:
# ── Loss + PSNR ───────────────────────────────────────────────────────────────

def l1_loss(pred, target):
    return torch.mean(torch.abs(pred - target))


def calc_psnr(pred, target):
    with torch.no_grad():
        pred   = pred.clamp(0, 1)
        target = target.clamp(0, 1)
        mse    = torch.mean((pred - target) ** 2)
        if mse == 0:
            return 100.0
        return 10.0 * math.log10(1.0 / mse.item())


# ── EMA ───────────────────────────────────────────────────────────────────────
class EMA:
    """
    Exponential Moving Average of model weights.
    Improves generalization and smooths training.
    """
    def __init__(self, model, decay=0.999):
        self.model  = model
        self.decay  = decay
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    def update(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = self.decay * self.shadow[n] + (1 - self.decay) * p.data

    def apply_shadow(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone()
                p.data = self.shadow[n]

    def restore(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                p.data = self.backup[n]


# ── Optimizer + Scheduler ─────────────────────────────────────────────────────
LR_INIT = 3e-4
LR_MIN  = 6e-9

optimizer = optim.Adam(
    model.parameters(),
    lr=LR_INIT,
    betas=(0.9, 0.999),
    weight_decay=0
)

# Cosine annealing with warmup
total_steps = NUM_EPOCHS * n_batches
warmup_steps = 10 * n_batches

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return LR_MIN / LR_INIT + 0.5 * (1 - LR_MIN / LR_INIT) * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

ema    = EMA(model, decay=0.999)
scaler = torch.amp.GradScaler('cuda', enabled=True, init_scale=1024)

print(f'Optimizer  : Adam (lr={LR_INIT})')
print(f'Scheduler  : Cosine with {10}-epoch warmup')
print(f'Loss       : L1')
print(f'EMA decay  : 0.999')

Optimizer  : Adam (lr=0.0003)
Scheduler  : Cosine with 10-epoch warmup
Loss       : L1
EMA decay  : 0.999


In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────

best_psnr = 0.0

print('=' * 60)
print('  Starting LiteSRNet Training')
print('  Target: PSNR > 24.3 dB | Low NPU Latency')
print('  Baseline: 23.1 dB (val) | 23.3 dB (test)')
print('=' * 60)
print()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    ep_loss = 0.0
    t0 = time.time()

    for lr_imgs, hr_imgs in train_loader:
        lr_imgs = lr_imgs.to(device, non_blocking=True)
        hr_imgs = hr_imgs.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            preds = model(lr_imgs)

        pc   = torch.clamp(preds.float(), 0, 1)
        loss = l1_loss(pc, hr_imgs.float())

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema.update()
        ep_loss += loss.item()

    current_lr = optimizer.param_groups[0]['lr']

    # Validate every 10 epochs + first 5 + last 20
    if epoch <= 5 or epoch % 10 == 0 or epoch > NUM_EPOCHS - 20:
        ema.apply_shadow()
        model.eval()
        ps, c = 0.0, 0
        with torch.no_grad():
            for l, h in val_loader:
                l = l.to(device)
                h = h.to(device)
                pr = model(l)
                for i in range(pr.size(0)):
                    ps += calc_psnr(pr[i], h[i])
                    c  += 1
        mp  = ps / max(c, 1)
        tgt = ' >> TARGET' if mp > 24.3 else ''
        print(f'Ep [{epoch:3d}/{NUM_EPOCHS}]  '
              f'loss={ep_loss/max(n_batches,1):.5f}  '
              f'PSNR={mp:.2f} dB{tgt}  '
              f'lr={current_lr:.2e}')

        if mp > best_psnr:
            best_psnr = mp
            torch.save({
                'epoch'            : epoch,
                'model_state_dict' : model.state_dict(),
                'ema_shadow'       : ema.shadow,
                'best_psnr'        : best_psnr
            }, os.path.join(CHECKPOINT_DIR, 'best_model.pth'))

        ema.restore()
        model.train()

print()
print('=' * 60)
print(f'Training complete! Best PSNR: {best_psnr:.2f} dB')
print(f'Target 24.3 dB: {"PASSED" if best_psnr > 24.3 else "NOT reached"}')
print('=' * 60)

  Starting LiteSRNet Training
  Target: PSNR > 24.3 dB | Low NPU Latency
  Baseline: 23.1 dB (val) | 23.3 dB (test)

Ep [  1/350]  loss=0.04629  PSNR=23.11 dB  lr=3.00e-05
Ep [  2/350]  loss=0.04598  PSNR=23.08 dB  lr=6.00e-05
Ep [  3/350]  loss=0.04504  PSNR=23.07 dB  lr=9.00e-05
Ep [  4/350]  loss=0.04443  PSNR=23.09 dB  lr=1.20e-04
Ep [  5/350]  loss=0.04403  PSNR=23.11 dB  lr=1.50e-04
Ep [ 10/350]  loss=0.04293  PSNR=23.26 dB  lr=3.00e-04
Ep [ 20/350]  loss=0.04184  PSNR=23.57 dB  lr=2.99e-04
Ep [ 30/350]  loss=0.04149  PSNR=23.69 dB  lr=2.97e-04
Ep [ 40/350]  loss=0.04127  PSNR=23.70 dB  lr=2.94e-04
Ep [ 50/350]  loss=0.04102  PSNR=23.72 dB  lr=2.90e-04
Ep [ 60/350]  loss=0.04074  PSNR=23.80 dB  lr=2.84e-04
Ep [ 70/350]  loss=0.04059  PSNR=23.78 dB  lr=2.78e-04
Ep [ 80/350]  loss=0.04062  PSNR=23.83 dB  lr=2.70e-04
Ep [ 90/350]  loss=0.04060  PSNR=23.97 dB  lr=2.61e-04
Ep [100/350]  loss=0.04035  PSNR=24.01 dB  lr=2.51e-04
Ep [110/350]  loss=0.04035  PSNR=24.08 dB  lr=2.40e-04
Ep 

In [ ]:
# ── Export ONNX (opset 11) ────────────────────────────────────────────────────
import onnx

print('Loading best weights for ONNX export...')
ckpt = torch.load(
    os.path.join(CHECKPOINT_DIR, 'best_model.pth'),
    map_location=device
)

# Apply EMA weights
for n, p in model.named_parameters():
    if n in ckpt.get('ema_shadow', {}):
        p.data = ckpt['ema_shadow'][n].to(device)

model.eval()
model_cpu  = model.cpu()
dummy_cpu  = torch.randn(1, 3, 256, 256)

# Export with legacy exporter to get opset 11
torch.onnx.export(
    model_cpu,
    dummy_cpu,
    ONNX_PATH,
    input_names=['input'],
    output_names=['output'],
    opset_version=11,
    do_constant_folding=True,
    dynamo=False
)

# Merge into single file
onnx_model = onnx.load(ONNX_PATH)
onnx.save_model(onnx_model, ONNX_PATH, save_as_external_data=False)

# Verify
onnx.checker.check_model(onnx.load(ONNX_PATH))
loaded  = onnx.load(ONNX_PATH)
sz_mb   = os.path.getsize(ONNX_PATH) / 1e6
print(f'ONNX exported successfully')
print(f'Opset   : {loaded.opset_import[0].version}')
print(f'Size    : {sz_mb:.2f} MB')

# Save to Drive
shutil.copy2(ONNX_PATH, os.path.join(DRIVE_SAVE, 'sr_model.onnx'))
shutil.copy2(
    os.path.join(CHECKPOINT_DIR, 'best_model.pth'),
    os.path.join(DRIVE_SAVE, 'best_model.pth')
)

print(f'Saved to Drive: {DRIVE_SAVE}')

model = model_cpu.to(device)

Loading best weights for ONNX export...


/tmp/ipykernel_1100/902846780.py:20: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX exported successfully
Opset   : 11
Size    : 2.33 MB
Saved to Drive: /content/drive/MyDrive/Lab3/outputs_litesr


In [ ]:
# ── Calibration Images ────────────────────────────────────────────────────────
CALIB_DIR = os.path.join(DRIVE_SAVE, 'calibration_images')
os.makedirs(CALIB_DIR, exist_ok=True)

N_CALIB     = 50
PER_ZIP     = 10
calib_count = 0

for lr_root in lr_train_roots:
    files = find_images(lr_root)
    step  = max(1, len(files) // PER_ZIP)
    for fpath in files[::step][:PER_ZIP]:
        img = Image.open(fpath).convert('RGB').resize((256, 256), Image.BILINEAR)
        img.save(os.path.join(CALIB_DIR, f'calib_{calib_count:03d}.jpg'))
        calib_count += 1

print(f'Saved {calib_count} calibration images to: {CALIB_DIR}')

Saved 50 calibration images to: /content/drive/MyDrive/Lab3/outputs_litesr/calibration_images


In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
n_params = sum(p.numel() for p in model.parameters())
sz_mb    = os.path.getsize(ONNX_PATH) / 1e6

print('=' * 60)
print('  LAB 3 - COMPLETE')
print('=' * 60)
print(f'Architecture  : LiteSRNet (ch=56, blocks=9)')
print(f'Parameters    : {n_params:,}')
print(f'Best PSNR     : {best_psnr:.2f} dB')
print(f'Target 24.3   : {"PASSED" if best_psnr > 24.3 else "check test data"}')
print(f'ONNX size     : {sz_mb:.2f} MB')
print(f'GPU Latency   : {gpu_lat:.2f} ms/image')
print(f'Est. NPU      : ~{gpu_lat*5:.1f} to {gpu_lat*6:.1f} ms/image')
print()
print('GRADING SCALE:')
print('  PSNR > 24.3 + Latency < 6ms   -> 18 pts')
print('  PSNR > 24.3 + Latency < 8ms   -> 17 pts')
print('  PSNR > 24.3 + Latency < 10ms  -> 16 pts')
print('  PSNR > 24.3 + Latency < 12ms  -> 15 pts')
print('  PSNR > 24.3 + Latency < 14ms  -> 14 pts')
print('  23.3-24.3   + Any             -> 8 pts')
print()
print('CANVAS SUBMISSION:')
print('  - Training notebook (.ipynb)')
print('  - onnx_to_mxq.py')
print('  - sr_model.mxq')
print('  - best_model.pth')
print('=' * 60)

  LAB 3 - COMPLETE
Architecture  : LiteSRNet (ch=56, blocks=9)
Parameters    : 581,339
Best PSNR     : 24.33 dB
Target 24.3   : PASSED
ONNX size     : 2.33 MB
GPU Latency   : 1.79 ms/image
Est. NPU      : ~8.9 to 10.7 ms/image

GRADING SCALE:
  PSNR > 24.3 + Latency < 6ms   -> 18 pts
  PSNR > 24.3 + Latency < 8ms   -> 17 pts
  PSNR > 24.3 + Latency < 10ms  -> 16 pts
  PSNR > 24.3 + Latency < 12ms  -> 15 pts
  PSNR > 24.3 + Latency < 14ms  -> 14 pts
  23.3-24.3   + Any             -> 8 pts

CANVAS SUBMISSION:
  - Training notebook (.ipynb)
  - onnx_to_mxq.py
  - sr_model.mxq
  - best_model.pth
